In [111]:
import pandas as pd
import os
import re

# ----------------------------
# SETTINGS
# ----------------------------
OHCHR_CSV = "./ohchr_resolutions.csv"
RESOLUTIONS_CSV = "./ga_resolutions_1946_2019.csv"
CITES_CSV = "./ga_citations_1946_2019.csv"

# ----------------------------
# LOAD DATA
# ----------------------------
ohchr_df = None
if not os.path.isfile(OHCHR_CSV):
    print(f"Error: File '{OHCHR_CSV}' does not exist.")

else:
    print("File found. Loading...")

    try:
        # Load in chunks to avoid memory issues
        chunksize = 2000
        chunks = []

        for chunk in pd.read_csv(OHCHR_CSV, chunksize=chunksize):
            chunks.append(chunk)

        ohchr_df = pd.concat(chunks, ignore_index=True)
        print("CSV file loaded successfully.")

    except Exception as e:
        print(f"Error loading CSV file: {e}")


File found. Loading...
CSV file loaded successfully.


In [112]:
df = None
if not os.path.isfile(RESOLUTIONS_CSV):
    print(f"Error: File '{RESOLUTIONS_CSV}' does not exist.")
else:
    print("File found. Loading...")

    try:
        # Load in chunks to avoid memory issues
        chunksize = 2000
        chunks = []

        for chunk in pd.read_csv(RESOLUTIONS_CSV, chunksize=chunksize):
            chunks.append(chunk)

        df = pd.concat(chunks, ignore_index=True)
        print("CSV file loaded successfully.")

    except Exception as e:
        print(f"Error loading CSV file: {e}")

df['date_p'] = pd.to_datetime(df['date_p'], errors='coerce')

df = df[(df['date_p'] >= '2006-01-01') & (df['date_p'] <= '2019-12-31')]

File found. Loading...
CSV file loaded successfully.


In [113]:
cites_df = None
if not os.path.isfile(CITES_CSV):
    print(f"Error: File '{CITES_CSV}' does not exist.")
else:
    print("File found. Loading...")

    try:
        # Load in chunks to avoid memory issues
        chunksize = 2000
        chunks = []

        for chunk in pd.read_csv(CITES_CSV, chunksize=chunksize):
            chunks.append(chunk)

        cites_df = pd.concat(chunks, ignore_index=True)
        print("CSV file loaded successfully.")

    except Exception as e:
        print(f"Error loading CSV file: {e}")

File found. Loading...
CSV file loaded successfully.


In [114]:
# Session number with GA means general assembly, we are not interested in those: filter out
ohchr_df = ohchr_df[~ohchr_df['Session number'].str.contains('GA')]

# The resolutions go from 1946 to 2019
ohchr_df = ohchr_df[ohchr_df["Year"] <= 2019]

# Remove also the President statement
ohchr_df = ohchr_df[~ohchr_df['Text type'].str.lower().str.contains('presidential', na=False)]

# Remove the Decisions
ohchr_df = ohchr_df[~ohchr_df['Text type'].str.lower().str.contains('decision', na=False)]


In [115]:
import re
print(f"Length of ohchr_df: {len(ohchr_df)}") #1153
not_in_cites = ohchr_df[~ohchr_df['Text number'].isin(cites_df['res_id2_unlet_recv'].unique())].copy()
ohchr_res = not_in_cites['Text number'].str.lower().unique()
ohchr_res = ' ' + ohchr_res + ' '

print(f"Length of 'secure' ohchr_df: {len(not_in_cites)}")

res_giv, res_rec, contents = [], [], []

for res_id in ohchr_res:
    for ga_res, content in zip(df['res_id2'], df['content']):

        if res_id not in content:
            continue

        if 'human rights council' not in content.lower():
            continue

        sentences = re.split(r'(?<=[.])\s+', content)

        found = False
        for sent in sentences:
            sent_low = sent.lower()

            if res_id in sent and 'human rights council' in sent_low:
                found = True
                break

        if found:
            res_giv.append(ga_res)
            res_rec.append(res_id)
            contents.append(content)

new_cites = pd.DataFrame({
    'res_giv': res_giv,
    'content': contents,
    'res_rec': res_rec
})


print(new_cites.shape) #(4387, 3) hrcouncil in the same phrase (588, 3)

Length of ohchr_df: 1153
Length of 'secure' ohchr_df: 896
(588, 3)


In [116]:
# Now we scan for the ones that share code with the normal General Assembly resolution codes
print(f"Length of ohchr_df: {len(ohchr_df)}") #1153
in_cites = ohchr_df[ohchr_df['Text number'].isin(cites_df['res_id2_unlet_recv'].unique())].copy()
ohchr_res = in_cites['Text number'].str.lower().unique()
ohchr_res = ' ' + ohchr_res + ' '

print(f"Length of resolutions in ohchr_df that are shared with UNGA resolution codes: {len(in_cites)}")

res_giv, res_rec, contents = [], [], []

for res_id in ohchr_res:
    for ga_res, content in zip(df['res_id2'], df['content']):

        if res_id not in content:
            continue

        if 'human rights council' not in content.lower():
            continue

        sentences = re.split(r'(?<=[.])\s+', content)

        found = False
        for sent in sentences:
            sent_low = sent.lower()

            if res_id in sent and 'human rights council' in sent_low:
                found = True
                break

        if found:
            res_giv.append(ga_res)
            res_rec.append(res_id)
            contents.append(content)

to_review = pd.DataFrame({
    'res_giv': res_giv,
    'content': contents,
    'res_rec': res_rec
})

print(to_review.shape)

Length of ohchr_df: 1153
Length of resolutions in ohchr_df that are shared with UNGA resolution codes: 257
(74, 3)


In [117]:
import pandas as pd
import re

PADDING = 120

results = []

for res_giv, grp in to_review.groupby("res_giv"):

    content = grp["content"].iloc[0]

    matches = []

    for res_rec in grp["res_rec"].dropna().unique():

        # escape in case res_rec contains regex characters
        for m in re.finditer(re.escape(str(res_rec)), content):
            matches.append({
                "res_rec": res_rec,
                "start": m.start(),
                "end": m.end()
            })

    if not matches:
        continue

    # earliest and latest occurrence
    first_start = min(m["start"] for m in matches)
    last_end = max(m["end"] for m in matches)

    window_start = max(0, first_start - PADDING)
    window_end = min(len(content), last_end + PADDING)

    window_text = content[window_start:window_end]

    detected_res_rec = sorted(set(m["res_rec"] for m in matches))

    results.append({
        "res_giv": res_giv,
        "window": window_text,
        "detected_res_rec": detected_res_rec
    })

windows_df = pd.DataFrame(results)

for _, row in windows_df.iterrows():
    print("=" * 10)
    print("res_giv:", row["res_giv"])
    print("\ndetected_res_rec:")
    print(row["detected_res_rec"])
    print("\nwindow:")
    print(row["window"])
    print()

res_giv: 66/253 b

detected_res_rec:
[' 42/37 ']

window:
1 march 2012, 19/22 of 23 march 2012, s-19/1 of 1 june 2012 and 20/22 of 6 july 2012, and recalling also its resolutions 42/37 a of 30 november 1987, 42/37 b of 30 november 1987, 42/37 c of 30 november 1987, 43/74 a of 7 december 1988, 43/74 b of 7 december 1988, 43/74 c of 7 december 1988 and 66/35 of 2

res_giv: 67/173

detected_res_rec:
[' 39/11 ']

window:
hts council resolution 20/15 of 5 july 2012, entitled “promotion of the right to peace”, 1 recalling also its resolution 39/11 of 12 november 1984, entitled “declaration on the right of peoples to peace”, and the united nations millennium declarat

res_giv: 69/176

detected_res_rec:
[' 39/11 ']

window:
13 june 20132 and 27/17 of 25 september 2014,3 entitled “promotion of the right to peace”, recalling also its resolution 39/11 of 12 november 1984, entitled “declaration on the right of peoples to peace”, and the united nations millennium declarat

res_giv: 71/130

detected

After a close inspection, these codes must be deleted from the to_review dataframe.
66/253 b
67/173
69/176

In [125]:
PADDING = 120

results = []

for res_giv, grp in new_cites.groupby("res_giv"):

    content = grp["content"].iloc[0]

    matches = []

    for res_rec in grp["res_rec"].dropna().unique():

        # escape in case res_rec contains regex characters
        for m in re.finditer(re.escape(str(res_rec)), content):
            matches.append({
                "res_rec": res_rec,
                "start": m.start(),
                "end": m.end()
            })

    if not matches:
        continue

    # earliest and latest occurrence
    first_start = min(m["start"] for m in matches)
    last_end = max(m["end"] for m in matches)

    window_start = max(0, first_start - PADDING)
    window_end = min(len(content), last_end + PADDING)

    window_text = content[window_start:window_end]

    detected_res_rec = sorted(set(m["res_rec"] for m in matches))

    results.append({
        "res_giv": res_giv,
        "window": window_text,
        "detected_res_rec": detected_res_rec
    })

windows_df = pd.DataFrame(results)
print(len(windows_df))
for _, row in windows_df.iterrows():
    print("=" * 10)
    print("res_giv:", row["res_giv"])
    print("\ndetected_res_rec:")
    print(row["detected_res_rec"])
    print("\nwindow:")
    print(row["window"])
    print()

287
res_giv: 61/149

detected_res_rec:
[' 1/5 ']

window:
related intolerance in order to address multiple forms of discrimination, taking note of human rights council resolution 1/5 of 30 june 2006, 2 taking note also of commission on human rights resolutions 2002/68 of 25 april 2002, 3 2003/30 of 23 

res_giv: 61/154

detected_res_rec:
[' s-2/1 ']

window:
nd the statement by the president of the council of 30 july 2006, 9 bearing in mind also human rights council resolution s-2/1 entitled “the grave situation of human rights in lebanon caused by israeli military operations”, adopted by the council 

res_giv: 61/169

detected_res_rec:
[' 1/4 ']

window:
t, particularly of developing countries”, 5 recalling also all its previous resolutions, human rights council resolution 1/4 of 30 june 2006 6 and those of the commission on human rights on the right to development, in particular commission reso

res_giv: 61/177

detected_res_rec:
[' 1/1 ']

window:
otection of all persons from enforc

In [126]:
# Remove the special 3 cases where human right council resolution shouldn't be considered and save the cite

# Concat
all_cites = pd.concat([new_cites, to_review], ignore_index=True)

all_cites = all_cites[
    ~(
        ((all_cites["res_giv"] == "66/253 b") & (all_cites["res_rec"] == "42/37")) |
        ((all_cites["res_giv"] == "67/173") & (all_cites["res_rec"] == "39/11")) |
        ((all_cites["res_giv"] == "69/176") & (all_cites["res_rec"] == "39/11"))
    )
].reset_index(drop=True)

print(all_cites.shape)


(662, 3)


In [127]:
all_cites["res_rec"] = "A/HRC/RES/" + all_cites["res_rec"].astype(str)
all_cites.rename({'res_giv': 'res_id2', 'res_rec': 'ohchr_resolution'}, axis=1, inplace=True)

# ----------------------------
# SAVE DATA
# ----------------------------
all_cites.to_csv("./ohchr_citations.csv", index=False)